# BirdQuest Rarity Scoring Prototype

This notebook:
1. Loads the eBird Basic Dataset (GB subset)
2. Computes species frequency tables by (H3 cell, month)
3. Implements the rarity scoring formula
4. Tests it on 20 hand-crafted sightings

## eBird Data Access

You need to request access to the eBird Basic Dataset:
1. Go to https://ebird.org/data/download
2. Sign in (or create a free eBird account)
3. Request access — it's auto-approved within ~24 hours
4. Download the **GB (United Kingdom)** subset
5. Place the file at `data/ebd_GB_relMay-2026.txt.gz`

The file is a gzipped tab-separated dataset, typically 1-2 GB for GB.

In [ ]:
import pandas as pd
import h3
from pathlib import Path
from datetime import datetime

DATA_DIR = Path("data")
EBD_FILE = DATA_DIR / "ebd_GB_relMay-2026.txt.gz"
BASE_VALUES_FILE = DATA_DIR / "base_values.csv"
H3_RESOLUTION = 6  # ~36 km² hexagons
MIN_CHECKLISTS_THRESHOLD = 50  # below this, fall back to coarser resolution

## 1. Load eBird data (chunked)

The eBird Basic Dataset has these key columns:
- `SPECIES CODE` — 6-letter eBird code
- `COMMON NAME`
- `OBSERVATION DATE` — YYYY-MM-DD
- `LATITUDE`, `LONGITUDE`
- `SAMPLING EVENT IDENTIFIER` — unique checklist ID
- `OBSERVATION COUNT` — count or 'X' for presence

We only need a subset of columns. We'll read in chunks to manage memory.

In [ ]:
COLS_TO_LOAD = [
    "SPECIES CODE",
    "COMMON NAME",
    "OBSERVATION DATE",
    "LATITUDE",
    "LONGITUDE",
    "SAMPLING EVENT IDENTIFIER",
]

def load_ebd_chunked(filepath: Path, chunksize: int = 500_000) -> pd.DataFrame:
    """Load eBird Basic Dataset in chunks, keeping only needed columns."""
    chunks = []
    reader = pd.read_csv(
        filepath,
        sep="\t",
        usecols=COLS_TO_LOAD,
        dtype={"SPECIES CODE": str, "COMMON NAME": str, "SAMPLING EVENT IDENTIFIER": str},
        parse_dates=["OBSERVATION DATE"],
        compression="gzip",
        chunksize=chunksize,
        low_memory=False,
    )
    for i, chunk in enumerate(reader):
        # Drop rows with missing coordinates
        chunk = chunk.dropna(subset=["LATITUDE", "LONGITUDE"])
        chunks.append(chunk)
        print(f"  Loaded chunk {i+1}: {len(chunk):,} rows")
    
    df = pd.concat(chunks, ignore_index=True)
    print(f"\nTotal rows loaded: {len(df):,}")
    return df

# --- Uncomment when you have the data file ---
# ebd = load_ebd_chunked(EBD_FILE)
# ebd.head()

## 2. Compute H3 cells and frequency tables

In [ ]:
def add_h3_and_month(df: pd.DataFrame, resolution: int = H3_RESOLUTION) -> pd.DataFrame:
    """Add h3_cell and month columns."""
    df = df.copy()
    df["h3_cell"] = [
        h3.latlng_to_cell(lat, lng, resolution)
        for lat, lng in zip(df["LATITUDE"], df["LONGITUDE"])
    ]
    df["month"] = df["OBSERVATION DATE"].dt.month
    return df


def compute_frequency_table(df: pd.DataFrame) -> pd.DataFrame:
    """Compute species frequency per (h3_cell, month).
    
    Frequency = (number of checklists reporting species) / (total checklists in cell+month)
    """
    # Total unique checklists per (h3_cell, month)
    total_checklists = (
        df.groupby(["h3_cell", "month"])["SAMPLING EVENT IDENTIFIER"]
        .nunique()
        .rename("total_checklists")
    )
    
    # Checklists reporting each species per (h3_cell, month)
    species_checklists = (
        df.groupby(["h3_cell", "month", "SPECIES CODE"])["SAMPLING EVENT IDENTIFIER"]
        .nunique()
        .rename("species_checklists")
        .reset_index()
    )
    
    # Join and compute frequency
    freq = species_checklists.merge(
        total_checklists.reset_index(), on=["h3_cell", "month"]
    )
    freq["frequency"] = freq["species_checklists"] / freq["total_checklists"]
    
    return freq[["h3_cell", "month", "SPECIES CODE", "frequency", "total_checklists"]]

# --- Uncomment when you have the data file ---
# ebd = add_h3_and_month(ebd)
# freq_table = compute_frequency_table(ebd)
# freq_table.to_parquet(DATA_DIR / "frequency_table.parquet", index=False)
# print(f"Frequency table: {len(freq_table):,} rows")
# freq_table.head(10)

## 3. Rarity scoring formula

In [ ]:
# Load base values
base_values = pd.read_csv(BASE_VALUES_FILE)
BASE_VALUE_MAP = dict(zip(base_values["species_code"], base_values["base_value"]))
print(f"Loaded {len(BASE_VALUE_MAP)} species base values")
base_values.head()

In [ ]:
def get_expectation_multiplier(frequency: float | None) -> tuple[float, str]:
    """Map checklist frequency to a multiplier and tier label."""
    if frequency is None or frequency == 0:
        return 50.0, "zero_records"
    elif frequency < 0.001:
        return 25.0, "<0.1%"
    elif frequency < 0.01:
        return 10.0, "0.1-1%"
    elif frequency < 0.10:
        return 3.0, "1-10%"
    elif frequency < 0.50:
        return 1.0, "10-50%"
    else:
        return 0.5, ">50%"


def get_verification_multiplier(verified: bool) -> float:
    return 1.0 if verified else 0.5


def lookup_frequency(
    freq_table: pd.DataFrame,
    species_code: str,
    lat: float,
    lng: float,
    month: int,
) -> tuple[float | None, int, str]:
    """Look up frequency for a species at a location+month.
    
    Falls back to coarser H3 resolutions if the cell has too few checklists,
    then to country-level.
    
    Returns: (frequency, total_checklists, resolution_used)
    """
    for res in [H3_RESOLUTION, H3_RESOLUTION - 1, H3_RESOLUTION - 2]:
        cell = h3.latlng_to_cell(lat, lng, res)
        
        if res < H3_RESOLUTION:
            # For coarser resolutions, we need to find all child cells in the freq table
            # that fall within this parent cell
            children = set(h3.cell_to_children(cell, H3_RESOLUTION))
            mask = (
                freq_table["h3_cell"].isin(children)
                & (freq_table["month"] == month)
                & (freq_table["SPECIES CODE"] == species_code)
            )
            matches = freq_table[mask]
            if not matches.empty:
                total_cl = freq_table[
                    freq_table["h3_cell"].isin(children) & (freq_table["month"] == month)
                ]["total_checklists"].sum()
                if total_cl >= MIN_CHECKLISTS_THRESHOLD:
                    # Weighted average frequency across child cells
                    weighted_freq = (
                        (matches["frequency"] * matches["total_checklists"]).sum() / total_cl
                    )
                    return weighted_freq, int(total_cl), f"h3_res{res}"
        else:
            mask = (
                (freq_table["h3_cell"] == cell)
                & (freq_table["month"] == month)
                & (freq_table["SPECIES CODE"] == species_code)
            )
            matches = freq_table[mask]
            if not matches.empty:
                row = matches.iloc[0]
                if row["total_checklists"] >= MIN_CHECKLISTS_THRESHOLD:
                    return row["frequency"], int(row["total_checklists"]), f"h3_res{res}"
            else:
                # Species not found at this resolution — check if cell has enough data
                cell_data = freq_table[
                    (freq_table["h3_cell"] == cell) & (freq_table["month"] == month)
                ]
                if not cell_data.empty and cell_data.iloc[0]["total_checklists"] >= MIN_CHECKLISTS_THRESHOLD:
                    # Cell has data but species never recorded → frequency 0
                    return 0.0, int(cell_data.iloc[0]["total_checklists"]), f"h3_res{res}"
    
    # Country-level fallback
    country_mask = (
        (freq_table["month"] == month)
        & (freq_table["SPECIES CODE"] == species_code)
    )
    country_matches = freq_table[country_mask]
    if not country_matches.empty:
        total_cl = freq_table[freq_table["month"] == month]["total_checklists"].sum()
        weighted_freq = (
            (country_matches["frequency"] * country_matches["total_checklists"]).sum() / total_cl
        )
        return weighted_freq, int(total_cl), "country"
    
    return None, 0, "no_data"


def compute_points(
    species_code: str,
    lat: float,
    lng: float,
    observed_at: str | datetime,
    verified: bool,
    freq_table: pd.DataFrame,
    base_value_map: dict[str, int] | None = None,
) -> dict:
    """Compute rarity points for a sighting.
    
    Returns a dict with all intermediate values for debugging.
    """
    if base_value_map is None:
        base_value_map = BASE_VALUE_MAP
    
    if isinstance(observed_at, str):
        observed_at = datetime.fromisoformat(observed_at)
    
    month = observed_at.month
    base = base_value_map.get(species_code, 10)  # default 10 for unknown species
    
    frequency, total_checklists, resolution = lookup_frequency(
        freq_table, species_code, lat, lng, month
    )
    
    multiplier, tier = get_expectation_multiplier(frequency)
    verification_mult = get_verification_multiplier(verified)
    
    points = int(round(base * multiplier * verification_mult))
    
    return {
        "species_code": species_code,
        "lat": lat,
        "lng": lng,
        "month": month,
        "base_value": base,
        "frequency": frequency,
        "total_checklists": total_checklists,
        "resolution": resolution,
        "tier": tier,
        "local_multiplier": multiplier,
        "verification_mult": verification_mult,
        "points": points,
    }

## 4. Test with synthetic frequency data

Until you have the real eBird data, we use a synthetic frequency table to validate
the scoring formula. The synthetic data simulates realistic frequencies for well-known
UK birding locations.

In [ ]:
# --- Synthetic frequency table for testing ---
# We'll create fake frequency data for key locations so the formula can be tested
# before real eBird data is available.

import numpy as np

# Key test locations
LOCATIONS = {
    "hyde_park_london": (51.5073, -0.1657),
    "rspb_minsmere": (52.2388, 1.6180),
    "cairngorms": (57.0700, -3.6000),
    "norfolk_broads": (52.6200, 1.5000),
    "severn_estuary": (51.5500, -2.9000),
}

# Synthetic frequencies: (species_code, location_key, month, frequency)
# These approximate real birding patterns in the UK
SYNTHETIC_FREQ = [
    # Hyde Park — urban, common garden birds dominate
    ("houspa", "hyde_park_london", 5, 0.85),
    ("blabbi", "hyde_park_london", 5, 0.90),
    ("blutit", "hyde_park_london", 5, 0.75),
    ("gretit", "hyde_park_london", 5, 0.70),
    ("eurrob", "hyde_park_london", 5, 0.80),
    ("magpie", "hyde_park_london", 5, 0.65),
    ("carcar", "hyde_park_london", 5, 0.60),
    ("grechi", "hyde_park_london", 5, 0.15),
    ("eurspa", "hyde_park_london", 5, 0.05),
    ("perefl", "hyde_park_london", 5, 0.02),
    ("kinged", "hyde_park_london", 5, 0.008),
    ("combuz", "hyde_park_london", 5, 0.01),
    ("mallar", "hyde_park_london", 5, 0.55),
    ("canago", "hyde_park_london", 5, 0.40),
    ("mutswa", "hyde_park_london", 5, 0.30),
    # Hoopoe in Hyde Park in May — incredibly rare vagrant
    ("hoopoe", "hyde_park_london", 5, 0.0003),
    
    # RSPB Minsmere — premier birding site
    ("houspa", "rspb_minsmere", 5, 0.30),
    ("blabbi", "rspb_minsmere", 5, 0.85),
    ("eurrob", "rspb_minsmere", 5, 0.80),
    ("cuckoo", "rspb_minsmere", 5, 0.35),
    ("combuz", "rspb_minsmere", 5, 0.45),
    ("eurspa", "rspb_minsmere", 5, 0.20),
    ("kinged", "rspb_minsmere", 5, 0.12),
    ("lapwng", "rspb_minsmere", 5, 0.40),
    ("curlew", "rspb_minsmere", 5, 0.15),
    ("skylar", "rspb_minsmere", 5, 0.55),
    ("barnow", "rspb_minsmere", 5, 0.04),
    ("hoopoe", "rspb_minsmere", 5, 0.001),
    
    # Cairngorms — Scottish highlands
    ("blabbi", "cairngorms", 1, 0.40),
    ("eurrob", "cairngorms", 1, 0.35),
    ("combuz", "cairngorms", 1, 0.30),
    ("redkit", "cairngorms", 1, 0.10),
    ("golcre", "cairngorms", 1, 0.25),
    ("siskin", "cairngorms", 1, 0.20),
    ("waxwng", "cairngorms", 1, 0.005),  # Winter vagrant, rare
    
    # Norfolk Broads — wetlands
    ("mallar", "norfolk_broads", 3, 0.80),
    ("gryhea", "norfolk_broads", 3, 0.55),
    ("cormo1", "norfolk_broads", 3, 0.40),
    ("kinged", "norfolk_broads", 3, 0.18),
    ("curlew", "norfolk_broads", 3, 0.08),
    
    # Severn Estuary — wintering waders
    ("oyscat", "severn_estuary", 12, 0.60),
    ("curlew", "severn_estuary", 12, 0.45),
    ("lapwng", "severn_estuary", 12, 0.50),
    ("whoswn", "severn_estuary", 12, 0.02),
]

# Build the synthetic frequency DataFrame
rows = []
for species_code, loc_key, month, freq in SYNTHETIC_FREQ:
    lat, lng = LOCATIONS[loc_key]
    cell = h3.latlng_to_cell(lat, lng, H3_RESOLUTION)
    rows.append({
        "h3_cell": cell,
        "month": month,
        "SPECIES CODE": species_code,
        "frequency": freq,
        "total_checklists": 200,  # assume well-surveyed cells
    })

synthetic_freq_table = pd.DataFrame(rows)
print(f"Synthetic frequency table: {len(synthetic_freq_table)} rows")
synthetic_freq_table.head()

In [ ]:
# --- 20 test sightings ---

test_sightings = [
    # (description, species_code, lat, lng, date, verified)
    # Expected: boring common birds in cities = low points
    ("House Sparrow in Hyde Park, May", "houspa", 51.5073, -0.1657, "2026-05-15", True),
    ("Blackbird in Hyde Park, May", "blabbi", 51.5073, -0.1657, "2026-05-15", True),
    ("Mallard in Hyde Park, May", "mallar", 51.5073, -0.1657, "2026-05-15", True),
    ("Magpie in Hyde Park, May", "magpie", 51.5073, -0.1657, "2026-05-15", True),
    
    # Expected: mid-range — uncommon but not rare
    ("Great Spotted Woodpecker in Hyde Park, May", "grechi", 51.5073, -0.1657, "2026-05-15", True),
    ("Sparrowhawk in Hyde Park, May", "eurspa", 51.5073, -0.1657, "2026-05-15", True),
    ("Buzzard in Hyde Park, May", "combuz", 51.5073, -0.1657, "2026-05-15", True),
    
    # Expected: exciting sightings = high points
    ("Peregrine over Hyde Park, May", "perefl", 51.5073, -0.1657, "2026-05-15", True),
    ("Kingfisher at Hyde Park, May", "kinged", 51.5073, -0.1657, "2026-05-15", True),
    ("Hoopoe in Hyde Park, May (!)", "hoopoe", 51.5073, -0.1657, "2026-05-15", True),
    
    # Expected: common birds at a birding reserve = still low
    ("Blackbird at Minsmere, May", "blabbi", 52.2388, 1.6180, "2026-05-15", True),
    ("Skylark at Minsmere, May", "skylar", 52.2388, 1.6180, "2026-05-15", True),
    
    # Expected: good birds at a birding reserve = medium-high
    ("Cuckoo at Minsmere, May", "cuckoo", 52.2388, 1.6180, "2026-05-15", True),
    ("Kingfisher at Minsmere, May", "kinged", 52.2388, 1.6180, "2026-05-15", True),
    ("Barn Owl at Minsmere, May", "barnow", 52.2388, 1.6180, "2026-05-15", True),
    ("Hoopoe at Minsmere, May", "hoopoe", 52.2388, 1.6180, "2026-05-15", True),
    
    # Expected: Scottish winter specialities
    ("Waxwing in Cairngorms, Jan", "waxwng", 57.0700, -3.6000, "2026-01-15", True),
    ("Red Kite in Cairngorms, Jan", "redkit", 57.0700, -3.6000, "2026-01-15", True),
    
    # Expected: verification penalty
    ("Peregrine in Hyde Park, May (UNVERIFIED)", "perefl", 51.5073, -0.1657, "2026-05-15", False),
    
    # Expected: wintering waders at Severn Estuary
    ("Whooper Swan at Severn Estuary, Dec", "whoswn", 51.5500, -2.9000, "2026-12-15", True),
]

print(f"{len(test_sightings)} test sightings defined")

In [ ]:
# Run scoring on all test sightings
results = []
for desc, species, lat, lng, date, verified in test_sightings:
    result = compute_points(
        species_code=species,
        lat=lat,
        lng=lng,
        observed_at=date,
        verified=verified,
        freq_table=synthetic_freq_table,
    )
    result["description"] = desc
    results.append(result)

results_df = pd.DataFrame(results)

# Display the results table
display_cols = [
    "description", "species_code", "month", "frequency", "tier",
    "base_value", "local_multiplier", "verification_mult", "resolution", "points"
]
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_rows", 25)

results_df[display_cols]

In [ ]:
# Summary stats
print("Points distribution:")
print(f"  Min:    {results_df['points'].min()}")
print(f"  Max:    {results_df['points'].max()}")
print(f"  Mean:   {results_df['points'].mean():.1f}")
print(f"  Median: {results_df['points'].median():.1f}")
print()
print("Tier breakdown:")
print(results_df.groupby("tier")["points"].agg(["count", "mean", "min", "max"]).to_string())

## 5. Switch to real eBird data

Once you have the eBird Basic Dataset file, uncomment the cells in sections 1 and 2,
then re-run the notebook. Replace `synthetic_freq_table` with `freq_table` in the
scoring cell above.

```python
# Load real data
ebd = load_ebd_chunked(EBD_FILE)
ebd = add_h3_and_month(ebd)
freq_table = compute_frequency_table(ebd)
freq_table.to_parquet(DATA_DIR / "frequency_table.parquet", index=False)

# Then re-run scoring with freq_table instead of synthetic_freq_table
```